# **Proyecto Etapa 2. Construcción de Muestra Representativa — Particionamiento**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|----------|
| Ana Bonavides Aguilar | A01423281 |
|  | A01797775 |
|  | A01174130 |
|  | A01139580 |

## Descripción del Dataset

El dataset procesado en este notebook es el **GTEx Analysis V10 — Gene Expression TPM**, proveniente del proyecto Genotype-Tissue Expression (GTEx) del NIH / Broad Institute.

| Campo | Detalle |
|-------|--------|
| **Archivo principal** | `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct` |
| **Dimensiones** | 59,033 genes × 19,616 muestras de tejido humano |
| **Formato** | GCT 1.2 (TSV con 2 filas de metadatos al inicio) |
| **Metadatos de muestras** | `GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt` (48,231 filas) |
| **Metadatos de donantes** | `GTEx_Analysis_v10_Annotations_SubjectPhenotypesDS.txt` (981 donantes) |

## Reglas de Particionamiento

Las particiones se construyen combinando dos variables de caracterización:

- **Grupo de tejido** (5 valores): Nervioso, Hematopoyético, Cardiovascular, Musculoesquelético, Visceral/Metabólico  
- **Sexo biológico** (2 valores): Masculino, Femenino

Esto genera **10 particiones** con las siguientes probabilidades de ocurrencia calculadas sobre las 19,788 muestras RNASEQ con sexo conocido:

| # | Grupo de tejido | Sexo | N muestras | Probabilidad |
|---|----------------|------|-----------|-------------|
| P01 | Nervioso | Masculino | 2,838 | 0.1434 |
| P02 | Nervioso | Femenino | 1,066 | 0.0539 |
| P03 | Hematopoyético | Masculino | 932 | 0.0471 |
| P04 | Hematopoyético | Femenino | 475 | 0.0240 |
| P05 | Cardiovascular | Masculino | 1,573 | 0.0795 |
| P06 | Cardiovascular | Femenino | 771 | 0.0390 |
| P07 | Musculoesquelético | Masculino | 2,827 | 0.1429 |
| P08 | Musculoesquelético | Femenino | 1,349 | 0.0682 |
| P09 | Visceral/Metabólico | Masculino | 5,093 | 0.2574 |
| P10 | Visceral/Metabólico | Femenino | 2,864 | 0.1447 |
| | **TOTAL** | | **19,788** | **1.0000** |

### Mapeo de tejidos a grupos

| Grupo | Tejidos GTEx (SMTS) |
|-------|--------------------|
| Nervioso | Brain, Nerve |
| Hematopoyético | Blood, Bone Marrow, Spleen |
| Cardiovascular | Heart, Blood Vessel |
| Musculoesquelético | Muscle, Adipose Tissue, Skin |
| Visceral/Metabólico | Todo lo demás (Liver, Lung, Kidney, Esophagus, Colon, etc.) |

## 1. Imports y configuración de Spark

In [ ]:
import sys, os

# point Spark's JVM to the same Python running this notebook
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from functools import reduce

sys.path.insert(0, os.path.abspath('..'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    OUTPUT_DIR, N_GENES, SAMPLE_STEP, RANDOM_SEED
)

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'TPM file:           {FILE_PATH}')
print(f'Sample attributes:  {SAMPLE_ATTRS_PATH}')
print(f'Subject phenotypes: {SUBJECT_PHENO_PATH}')
print(f'Python:             {sys.executable}')

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("GTEx_Partitioning_TC5057") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

## 2. Carga de metadatos y construcción del índice de particiones

In [ ]:
# load sample attributes - keep only RNASEQ samples (the ones present in the TPM file)
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ')

# extract subject ID from sample ID: "GTEX-1117F-0005-SM-HL9SH" -> "GTEX-1117F"
sa_df = sa_df.withColumn(
    'SUBJID',
    F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1)
)

print(f'Muestras RNASEQ: {sa_df.count():,}')
sa_df.show(5, truncate=False)

In [ ]:
# load subject phenotypes
sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

print(f'Donantes: {sp_df.count():,}')
sp_df.show(5)

In [ ]:
# join to get tissue + sex per sample
meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

# map raw SMTS values to the 5 tissue groups
tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col)

# clean column name for joining: replace hyphens with underscores
meta_df = meta_df.withColumn(
    'COL_NAME',
    F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_')
)

print(f'Muestras con metadatos completos: {meta_df.count():,}')
meta_df.select('SAMPID', 'SMTS', 'TISSUE_GROUP', 'SEX_LABEL', 'COL_NAME').show(5, truncate=False)

In [ ]:
# verify partition sizes match expected values
print('Tamaño de cada partición:')
meta_df.groupBy('TISSUE_GROUP', 'SEX_LABEL') \
    .count() \
    .orderBy('TISSUE_GROUP', 'SEX_LABEL') \
    .show(20)

## 3. Carga del dataset TPM

Se carga el archivo GCT principal. Para mantener tiempos razonables en esta etapa de prueba, se toma 1 de cada `SAMPLE_STEP` columnas. En la etapa de entrenamiento/prueba se usarán todas las columnas.

In [ ]:
import pandas as pd
from pyspark.sql.functions import split as spark_split

# read column names with pandas (only header row — fast)
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names = peek.columns.tolist()

# sample every SAMPLE_STEP-th column
selected_indices = [0, 1] + list(range(2, len(all_col_names), SAMPLE_STEP))
selected_cols_raw = [all_col_names[i] for i in selected_indices]
clean_names = [c.replace('-', '_').replace('.', '_') for c in selected_cols_raw]
sample_cols = clean_names[2:]

print(f'Columnas de muestra seleccionadas: {len(sample_cols)} de {len(all_col_names) - 2:,}')

# use spark.read.text + pure Spark SQL to avoid Python RDD lambdas (Windows compatibility)
raw_df = spark.read.text(FILE_PATH)

# filter to gene rows using Spark column expression — no Python lambda
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))

# split each line on tab and pick selected columns by index
split_col = spark_split(F.col('value'), '\t')
df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(clean_names[idx]) for idx, i in enumerate(selected_indices)]
)

# cast sample columns to float
for c in sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM cargado: {df_tpm.count():,} genes x {len(clean_names)} columnas')
df_tpm.select(clean_names[:5]).show(3)

## 4. Función auxiliar de particionamiento

Dado un grupo de tejido y un sexo, devuelve el subconjunto de columnas del DataFrame TPM que pertenecen a esa partición, junto con `Name` y `Description`.

In [ ]:
# pre-build a dict: col_name -> (tissue_group, sex_label) for fast lookup
meta_lookup = {
    row['COL_NAME']: (row['TISSUE_GROUP'], row['SEX_LABEL'])
    for row in meta_df.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL').collect()
}

def get_partition_cols(tissue_group: str, sex_label: str) -> list:
    """Return column names in df_tpm that belong to the given partition."""
    return [
        c for c in sample_cols
        if meta_lookup.get(c) == (tissue_group, sex_label)
    ]

def get_partition(tissue_group: str, sex_label: str):
    """Return a DataFrame with Name, Description, and only the columns for this partition."""
    cols = get_partition_cols(tissue_group, sex_label)
    if not cols:
        print(f'[WARN] No hay columnas muestreadas para {tissue_group} + {sex_label}')
        return None
    return df_tpm.select(['Name', 'Description'] + cols)

print('Columnas disponibles por partición (en la muestra 1/SAMPLE_STEP):')
partitions = [
    ('Nervioso',            'Masculino'),
    ('Nervioso',            'Femenino'),
    ('Hematopoyetico',      'Masculino'),
    ('Hematopoyetico',      'Femenino'),
    ('Cardiovascular',      'Masculino'),
    ('Cardiovascular',      'Femenino'),
    ('Musculoesqueletico',  'Masculino'),
    ('Musculoesqueletico',  'Femenino'),
    ('Visceral_Metabolico', 'Masculino'),
    ('Visceral_Metabolico', 'Femenino'),
]
for tg, sx in partitions:
    n = len(get_partition_cols(tg, sx))
    print(f'  {tg:<25} + {sx:<12} -> {n} columnas')

## 5. Extracción de muestras por partición

Para cada una de las 10 particiones se muestra: dimensiones del subconjunto, los primeros genes y estadísticas básicas de TPM. **Estas son muestras de prueba** para verificar el funcionamiento del código; en la etapa posterior se usarán todas las columnas.

### P01 — Nervioso + Masculino

In [ ]:
p01 = get_partition('Nervioso', 'Masculino')
cols_p01 = get_partition_cols('Nervioso', 'Masculino')
print(f'P01 — Nervioso + Masculino: {N_GENES:,} genes x {len(cols_p01)} muestras (muestra 1/{SAMPLE_STEP})')
p01.select(['Name', 'Description'] + cols_p01[:3]).show(5, truncate=True)
p01.select(cols_p01[:3]).describe().show()

### P02 — Nervioso + Femenino

In [ ]:
p02 = get_partition('Nervioso', 'Femenino')
cols_p02 = get_partition_cols('Nervioso', 'Femenino')
print(f'P02 — Nervioso + Femenino: {N_GENES:,} genes x {len(cols_p02)} muestras')
p02.select(['Name', 'Description'] + cols_p02[:3]).show(5, truncate=True)
p02.select(cols_p02[:3]).describe().show()

### P03 — Hematopoyético + Masculino

In [ ]:
p03 = get_partition('Hematopoyetico', 'Masculino')
cols_p03 = get_partition_cols('Hematopoyetico', 'Masculino')
print(f'P03 — Hematopoyético + Masculino: {N_GENES:,} genes x {len(cols_p03)} muestras')
p03.select(['Name', 'Description'] + cols_p03[:3]).show(5, truncate=True)
p03.select(cols_p03[:3]).describe().show()

### P04 — Hematopoyético + Femenino

In [ ]:
p04 = get_partition('Hematopoyetico', 'Femenino')
cols_p04 = get_partition_cols('Hematopoyetico', 'Femenino')
print(f'P04 — Hematopoyético + Femenino: {N_GENES:,} genes x {len(cols_p04)} muestras')
p04.select(['Name', 'Description'] + cols_p04[:3]).show(5, truncate=True)
p04.select(cols_p04[:3]).describe().show()

### P05 — Cardiovascular + Masculino

In [ ]:
p05 = get_partition('Cardiovascular', 'Masculino')
cols_p05 = get_partition_cols('Cardiovascular', 'Masculino')
print(f'P05 — Cardiovascular + Masculino: {N_GENES:,} genes x {len(cols_p05)} muestras')
p05.select(['Name', 'Description'] + cols_p05[:3]).show(5, truncate=True)
p05.select(cols_p05[:3]).describe().show()

### P06 — Cardiovascular + Femenino

In [ ]:
p06 = get_partition('Cardiovascular', 'Femenino')
cols_p06 = get_partition_cols('Cardiovascular', 'Femenino')
print(f'P06 — Cardiovascular + Femenino: {N_GENES:,} genes x {len(cols_p06)} muestras')
p06.select(['Name', 'Description'] + cols_p06[:3]).show(5, truncate=True)
p06.select(cols_p06[:3]).describe().show()

### P07 — Musculoesquelético + Masculino

In [ ]:
p07 = get_partition('Musculoesqueletico', 'Masculino')
cols_p07 = get_partition_cols('Musculoesqueletico', 'Masculino')
print(f'P07 — Musculoesquelético + Masculino: {N_GENES:,} genes x {len(cols_p07)} muestras')
p07.select(['Name', 'Description'] + cols_p07[:3]).show(5, truncate=True)
p07.select(cols_p07[:3]).describe().show()

### P08 — Musculoesquelético + Femenino

In [ ]:
p08 = get_partition('Musculoesqueletico', 'Femenino')
cols_p08 = get_partition_cols('Musculoesqueletico', 'Femenino')
print(f'P08 — Musculoesquelético + Femenino: {N_GENES:,} genes x {len(cols_p08)} muestras')
p08.select(['Name', 'Description'] + cols_p08[:3]).show(5, truncate=True)
p08.select(cols_p08[:3]).describe().show()

### P09 — Visceral/Metabólico + Masculino

In [ ]:
p09 = get_partition('Visceral_Metabolico', 'Masculino')
cols_p09 = get_partition_cols('Visceral_Metabolico', 'Masculino')
print(f'P09 — Visceral/Metabólico + Masculino: {N_GENES:,} genes x {len(cols_p09)} muestras')
p09.select(['Name', 'Description'] + cols_p09[:3]).show(5, truncate=True)
p09.select(cols_p09[:3]).describe().show()

### P10 — Visceral/Metabólico + Femenino

In [ ]:
p10 = get_partition('Visceral_Metabolico', 'Femenino')
cols_p10 = get_partition_cols('Visceral_Metabolico', 'Femenino')
print(f'P10 — Visceral/Metabólico + Femenino: {N_GENES:,} genes x {len(cols_p10)} muestras')
p10.select(['Name', 'Description'] + cols_p10[:3]).show(5, truncate=True)
p10.select(cols_p10[:3]).describe().show()

## 6. Resumen de particiones

Tabla de verificación final con el conteo de columnas disponibles en la muestra 1/SAMPLE_STEP para cada partición.

In [ ]:
import pandas as pd

summary_rows = []
full_counts = {
    ('Nervioso',            'Masculino'): 2838,
    ('Nervioso',            'Femenino'):  1066,
    ('Hematopoyetico',      'Masculino'): 932,
    ('Hematopoyetico',      'Femenino'):  475,
    ('Cardiovascular',      'Masculino'): 1573,
    ('Cardiovascular',      'Femenino'):  771,
    ('Musculoesqueletico',  'Masculino'): 2827,
    ('Musculoesqueletico',  'Femenino'):  1349,
    ('Visceral_Metabolico', 'Masculino'): 5093,
    ('Visceral_Metabolico', 'Femenino'):  2864,
}
total = 19788

for i, (tg, sx) in enumerate(partitions, 1):
    n_full = full_counts[(tg, sx)]
    n_sampled = len(get_partition_cols(tg, sx))
    summary_rows.append({
        'Partición': f'P{i:02d}',
        'Grupo tejido': tg,
        'Sexo': sx,
        'N total': n_full,
        'Prob': round(n_full / total, 4),
        f'N muestra (1/{SAMPLE_STEP})': n_sampled,
    })

pd.DataFrame(summary_rows)

## Referencias

1. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
2. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex